# 🏔️ Collection Monitor & Gallery
Real-time monitoring and gallery browser for captured images.

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image, HTML, clear_output
import ipywidgets as widgets

log_path = Path('data/collection.log')

def get_job_captures():
    if not log_path.exists():
        return []
    
    captures = []
    with open(log_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                if entry.get('event') == 'CAPTURE' and entry.get('status') == 'SUCCESS':
                    meta = entry.get('metadata', {})
                    if 'image_path' in meta:
                        captures.append({
                            'timestamp': entry['timestamp'],
                            'path': meta['image_path'],
                            'step': meta.get('step_index', 'N/A'),
                            'metar': meta.get('metar_path')
                        })
            except:
                pass
    # Sort by timestamp descending (newest first)
    return sorted(captures, key=lambda x: x['timestamp'], reverse=True)

all_captures = get_job_captures()
current_index = 0

def show_review_mode(index):
    global current_index
    current_index = index
    clear_output(wait=True)
    
    cap = all_captures[index]
    img_path = Path(cap['path'])
    metar_path = Path(cap['metar']) if cap.get('metar') else None
    
    # Navigation Header
    btn_prev = widgets.Button(description="← Older", layout=widgets.Layout(width='100px'))
    btn_next = widgets.Button(description="Newer →", layout=widgets.Layout(width='100px'))
    btn_gall = widgets.Button(description="Back to Gallery", button_style='info', layout=widgets.Layout(width='150px'))
    
    btn_prev.on_click(lambda _: show_review_mode(index + 1) if index < len(all_captures) - 1 else None)
    btn_next.on_click(lambda _: show_review_mode(index - 1) if index > 0 else None)
    btn_gall.on_click(lambda _: show_gallery_mode())
    
    display(widgets.HBox([btn_prev, btn_next, btn_gall]))
    
    print(f"\nStep {cap['step']} | {cap['timestamp']}")
    if metar_path and metar_path.exists():
        print(f"METAR: {metar_path.read_text().strip()}")
    
    if img_path.exists():
        display(Image(filename=str(img_path), width=1000))
    else:
        print(f"Image not found: {img_path}")

def show_gallery_mode():
    clear_output(wait=True)
    display(HTML("<h2>🖼️ Capture Gallery</h2>"))
    
    if not all_captures:
        print("No captures found in logs.")
        return

    items = []
    for i, cap in enumerate(all_captures):
        img_path = Path(cap['path'])
        if not img_path.exists(): continue
        
        # Create Image widget for thumbnail
        with open(img_path, "rb") as f:
            image_widget = widgets.Image(
                value=f.read(),
                format='jpg',
                width=200,
                height=150,
            )
        
        # Create Review button
        btn = widgets.Button(
            description=f"Review Step {cap['step']}",
            layout=widgets.Layout(width='200px')
        )
        
        # Use a closure to capture the index correctly
        def make_click_handler(idx):
            return lambda _: show_review_mode(idx)
        
        btn.on_click(make_click_handler(i))
        
        # Combine into a VBox
        box = widgets.VBox([image_widget, btn], layout=widgets.Layout(margin='10px', align_items='center'))
        items.append(box)
    
    # Display in a GridBox
    grid = widgets.GridBox(items, layout=widgets.Layout(grid_template_columns="repeat(auto-fill, 220px)"))
    display(grid)

# Initial Load
show_gallery_mode()